# Multiple Linear Regression from Scratch: Closed-Form (Normal Equation) Solution

This notebook implements **Multiple Linear Regression** using the closed-form Normal Equation, derived and coded from first principles with NumPy. The custom implementation is then benchmarked against scikit-learn's `LinearRegression` to validate correctness.

**Dataset:** [Student Performance dataset](https://www.kaggle.com/datasets/nikhil7280/student-performance-multiple-linear-regression) — predicting a student's `Performance Index` from study habits, prior scores, sleep, and extracurricular activity.

**Workflow:**
1. Load and inspect the data
2. Encode categorical features and prepare the feature matrix
3. Split into train/test sets
4. Implement Multiple Linear Regression using the Normal Equation
5. Train the custom model and generate predictions
6. Validate against scikit-learn's `LinearRegression`
7. Compare learned parameters

In [1]:
# closed form solution multiple linear regression

## 1. Imports

Core libraries for numerical computation, data handling, and visualization.

In [114]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Load the Dataset

In [115]:
data = pd.read_csv("Student_Performance.csv")

Preview the first few rows to understand the structure and feature types.

In [116]:
data.head()

,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
0,7,99,Yes,9,1,91.0
1,4,82,No,4,2,65.0
2,8,51,Yes,7,2,45.0
3,5,52,Yes,5,2,36.0
4,7,75,No,8,5,66.0


## 3. Feature Preparation

### 3.1 Encoding the Categorical Feature

`Extracurricular Activities` is a binary categorical column (Yes/No). We use `OneHotEncoder` with `drop='first'` to avoid the dummy variable trap, producing a single numeric column.

In [117]:
from sklearn.preprocessing import OneHotEncoder

In [118]:
oe = OneHotEncoder(drop='first', sparse_output=False)

### 3.2 Train/Test Split Utility

Import scikit-learn's `train_test_split` for creating held-out evaluation data.

In [119]:
from sklearn.model_selection import train_test_split

### 3.3 Define Features and Target

Separate the predictors (`X`) from the target variable (`y`, the `Performance Index`).

In [120]:
X = data.drop(columns=['Performance Index'])
y = data['Performance Index']

Inspect the feature matrix `X`.

In [121]:
X

,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced
0,7,99,Yes,9,1
1,4,82,No,4,2
2,8,51,Yes,7,2
3,5,52,Yes,5,2
4,7,75,No,8,5
...,...,...,...,...,...
9995,1,49,Yes,4,2
9996,7,64,Yes,8,5
9997,6,83,Yes,8,5
9998,9,97,Yes,7,0


Inspect the target vector `y`.

In [122]:
y

,Performance Index
0,91.0
1,65.0
2,45.0
3,36.0
4,66.0
...,...
9995,23.0
9996,58.0
9997,74.0
9998,95.0


### 3.4 Fit and Apply the Encoder

Fit the encoder on the `Extracurricular Activities` column, then transform it into a numeric one-hot representation.

In [123]:
oe.fit(X[['Extracurricular Activities']])

OneHotEncoder(drop='first', sparse_output=False)

In [124]:
encoded = oe.transform(X[['Extracurricular Activities']])

Drop the original categorical column now that its encoded version has been computed.

In [125]:
X.drop(columns=['Extracurricular Activities'],inplace=True)

### 3.5 Assemble the Final Feature Matrix

Convert the remaining numeric features to a NumPy array and horizontally stack the one-hot encoded column back on.

In [126]:
X = X.to_numpy()

In [127]:
X = np.hstack((X,encoded))

Confirm the final shape of `X` — rows are samples, columns are features.

In [128]:
X.shape

(10000, 5)

Reshape `y` into a column vector so it aligns correctly for matrix operations in the Normal Equation.

In [129]:
y = y.to_numpy().reshape(-1,1)

## 4. Train/Test Split

Hold out 20% of the data for testing, using a fixed `random_state` for reproducibility.

In [130]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

Verify the resulting shapes of the train and test sets.

In [131]:
X_train.shape,X_test.shape,y_train.shape,y_test.shape

((8000, 5), (2000, 5), (8000, 1), (2000, 1))

Preview the training feature matrix.

In [132]:
X_train

array([[ 5., 49.,  7.,  5.,  0.],
       [ 2., 48.,  7.,  6.,  1.],
       [ 2., 81.,  7.,  2.,  0.],
       ...,
       [ 9., 48.,  7.,  6.,  0.],
       [ 1., 47.,  9.,  0.,  0.],
       [ 2., 46.,  6.,  6.,  0.]])

## 5. Multiple Linear Regression from Scratch

### The Normal Equation

For a linear model $y = X\beta + \epsilon$, the closed-form solution that minimizes the sum of squared residuals is:

$$\beta = (X^T X)^{-1} X^T y$$

where a column of ones is prepended to $X$ to account for the intercept term. Once $\beta$ is computed, the first element is the intercept and the remaining elements are the feature coefficients.

The class below implements this directly with NumPy — no gradient descent, no iterative optimization, just a direct matrix solution.

In [133]:
class MultipleLinearRegression:
  def __init__(self):
    self.coef_ = None
    self.intercept_ = None

  def fit(self,X_train,y_train):
    X_train = np.insert(X_train,0,1,axis=1)
    self.coef_ = np.linalg.inv(np.dot(X_train.T, X_train)).dot(X_train.T).dot(y_train)
    self.intercept_ = self.coef_[0]
    print(self.intercept_)
    self.coef_ = self.coef_[1:]
    print(self.coef_)
  def predict(self, X_test):
    return X_test @ self.coef_ + self.intercept_

### 5.1 Instantiate the Model

In [134]:
mlr = MultipleLinearRegression()

### 5.2 Fit the Model

Solve for the intercept and coefficients using the Normal Equation.

In [135]:
mlr.fit(X_train,y_train)

[-33.92194622]
[[2.85248393]
 [1.0169882 ]
 [0.47694148]
 [0.19183144]
 [0.60861668]]


### 5.3 Generate Predictions

Use the learned parameters to predict the `Performance Index` on the held-out test set.

In [136]:
mlr.predict(X_test)

array([[54.71185392],
       [22.61551294],
       [47.90314471],
       ...,
       [16.79341955],
       [63.34327368],
       [45.94262301]])

## 6. Validation Against Scikit-learn

To confirm the from-scratch implementation is mathematically correct, we fit scikit-learn's `LinearRegression` (which also solves the Normal Equation under the hood via least squares) on the same data.

In [137]:
from sklearn.linear_model import LinearRegression

In [138]:
lr = LinearRegression()

In [139]:
lr.fit(X_train,y_train)

LinearRegression()

Inspect scikit-learn's learned coefficients...

In [140]:
lr.coef_

array([[2.85248393, 1.0169882 , 0.47694148, 0.19183144, 0.60861668]])

...and its intercept.

In [141]:
lr.intercept_

array([-33.92194622])

## 7. Comparing Learned Parameters

Print the intercept and coefficients from both the custom implementation and scikit-learn side by side. If the Normal Equation was implemented correctly, these values should match closely (down to floating-point precision).

In [142]:
print(mlr.intercept_)
print(mlr.coef_)

print(lr.intercept_)
print(lr.coef_)

[-33.92194622]
[[2.85248393]
 [1.0169882 ]
 [0.47694148]
 [0.19183144]
 [0.60861668]]
[-33.92194622]
[[2.85248393 1.0169882  0.47694148 0.19183144 0.60861668]]


## Conclusion

The custom `MultipleLinearRegression` class — built entirely on the closed-form Normal Equation — produces intercept and coefficient values that match scikit-learn's optimized `LinearRegression` implementation. This confirms the mathematical derivation and NumPy implementation are correct, and demonstrates a solid first-principles understanding of how linear regression is solved analytically rather than through iterative optimization (e.g., gradient descent).

**Key takeaways:**
- The Normal Equation $\beta = (X^T X)^{-1} X^T y$ gives an exact, non-iterative solution for linear regression.
- Proper feature preparation (encoding categoricals, reshaping targets) is essential before applying matrix-based methods.
- Validating a from-scratch implementation against a trusted library is good practice for confirming correctness.